# Notebook #2b: ACRL Exploratory Data Analysis

<hr>

## 1.0 Project Dependencies

### 1.1 Pip Installs

### 1.2 Import Libraries

In [1]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
import plotly as py
import plotly.graph_objs as go
import plotly.express as px
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from plotly.subplots import make_subplots

### 1.3 Import Datasets

In [2]:
# ACRL
acrl_google       = pd.read_parquet("Cleaned Datasets/acrl_google")
acrl_indiv_emails = pd.read_parquet("Cleaned Datasets/acrl_indiv_emails")
acrl_syllabus     = pd.read_parquet("Cleaned Datasets/acrl_syllabus")
acrl_all_emails   = pd.read_parquet("Cleaned Datasets/acrl_all_emails")

# ALA Editions
editions_eblasts     = pd.read_parquet("Cleaned Datasets/editions_ala_eblasts.parquet")
editions_fw26_google = pd.read_parquet("Cleaned Datasets/editions_ala_fw26_google.parquet")
editions_ss25_google = pd.read_parquet("Cleaned Datasets/editions_ala_ss25_google.parquet")
editions_ala_stats   = pd.read_parquet("Cleaned Datasets/editions_ala_stats_eblasts")

### 1.4 Standardize Setup & Styling

In [3]:
ALA_BLUE   = '#003F72'
ALA_RED    = '#C8102E'
ACRL_TEAL  = '#00857C'
ACCENT     = '#F5A623'
LIGHT_GRAY = '#F4F4F4'

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   LIGHT_GRAY,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.family':      'sans-serif',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})

<hr>

## 2.0 ACRL Summary Statistics

### 2.1 ACRL Data

In [4]:
# acrl_google has a single aggregated row (Jan 2025 – Feb 2026)
display(acrl_google)

,date_start,date_end,total_views,active_users,views_per_active_user,avg_engagement_time_seconds_per_active_user,event_count,key_events,total_revenue
0,2025-01-01,2026-02-19,4784,3345,1.43,47,13754,0,0


In [5]:
acrl_indiv_emails

,Name,Subject,Sent,Folder,# Delivered,Delivered %,Total Opens,Open %,Total Clicks,Click %,Unsubs,Unsubs %,body_text,group,Subject_clean
0,Library Publishing Book,"New from ACRL - ""Library Publishing: How to La...",2025-10-29 11:14:00,ACRL,18350,99.8,6137,33.4,247,4.0,4,0.0,None,None,"New from ACRL - ""Library Publishing: How to La..."
1,Navigating Disability book,"New from ACRL - ""Navigating Disability in the ...",2025-10-09 11:14:00,ACRL,18408,99.8,6010,32.6,292,4.9,7,0.0,None,None,"New from ACRL - ""Navigating Disability in the ..."
2,MVBD book,"New from ACRL - ""Making Values-Based Decisions...",2025-09-16 11:14:00,ACRL,18211,99.9,5948,32.7,267,4.5,1,0.0,None,None,"New from ACRL - ""Making Values-Based Decisions..."
3,Valuing the CCL book,"New from ACRL - ""Valuing the Community College...",2025-07-30 11:14:00,ACRL,18314,99.9,6978,38.1,226,3.2,5,0.0,ACRL announces the publication of Valuing the ...,low,"New from ACRL - ""Valuing the Community College..."
4,Supporting Neurodiverse book,"New from ACRL - ""Supporting Neurodiverse Stude...",2025-06-24 11:14:00,ACRL,18420,99.9,7672,41.7,354,4.6,10,0.1,None,None,"New from ACRL - ""Supporting Neurodiverse Stude..."
5,Open Science Cookbook,"New from ACRL - ""The Open Science Cookbook""",2025-06-11 11:14:00,ACRL,18217,99.9,7833,43.0,686,8.8,0,0.0,ACRL announces the publication of The Open Sci...,high,"New from ACRL - ""The Open Science Cookbook"""
6,Legislative Advocacy Book,"New from ACRL - ""Legislative Advocacy and Publ...",2025-05-22 11:14:00,ACRL,18295,99.7,7129,39.0,957,13.4,11,0.1,ACRL announces the publication of Legislative ...,high,"New from ACRL - ""Legislative Advocacy and Publ..."
7,Instructional Design Book,"New from ACRL - ""Instructional Design for Teac...",2025-04-15 11:13:00,ACRL,18142,99.9,5069,27.9,395,7.8,5,0.0,None,None,"New from ACRL - ""Instructional Design for Teac..."
8,Integration Book,"New from ACRL - ""From Interrogation to Integra...",2025-04-09 11:14:00,ACRL,18153,99.9,5276,29.1,239,4.5,4,0.0,None,None,"New from ACRL - ""From Interrogation to Integra..."
9,Student Success Book,"New from ACRL - ""Student Success Librarianship...",2025-03-11 11:14:00,ACRL,17982,99.9,5212,29.0,560,10.7,3,0.0,ACRL announces the publication of Student Succ...,high,"New from ACRL - ""Student Success Librarianship..."


In [6]:
acrl_indiv_emails.describe()

,Sent,# Delivered,Delivered %,Total Opens,Open %,Total Clicks,Click %,Unsubs,Unsubs %
count,19,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,2025-03-19 17:48:06.315789568,18023.368421,99.873684,5381.421053,29.778947,373.263158,9.163158,4.052632,0.010526
min,2024-09-04 11:14:00,17664.000000,99.700000,623.000000,3.500000,223.000000,3.200000,0.000000,0.000000
25%,2024-11-27 00:26:30,17751.500000,99.900000,4835.500000,27.100000,261.500000,4.550000,2.000000,0.000000
50%,2025-03-11 11:14:00,17982.000000,99.900000,5212.000000,29.000000,319.000000,6.100000,4.000000,0.000000
75%,2025-06-17 23:14:00,18256.000000,99.900000,6073.500000,33.050000,383.500000,8.450000,5.000000,0.000000
max,2025-10-29 11:14:00,18420.000000,100.000000,7833.000000,43.000000,957.000000,54.900000,11.000000,0.100000
std,NaN,271.171576,0.073349,1572.278612,8.446799,183.229923,11.365504,2.990238,0.031530


In [7]:
acrl_syllabus

,Name,Subject,Sent,Folder,# Delivered,Total Opens,Open %,Total Clicks,Click %
0,Syllabus - February 2026,The Syllabus From ACRL - February 2026,2026-02-18 12:14:00,ACRL,22354,5054,13.80,696,13.80
1,Syllabus - January 2026,The Syllabus From ACRL - January 2026,2026-01-22 11:12:00,ACRL,22456,4852,21.60,646,13.30
2,Syllabus - December 2025,The Syllabus From ACRL - December 2025,2025-12-10 12:13:00,ACRL,22591,5108,22.60,670,13.10
3,Syllabus - November 2025,The Syllabus From ACRL - November 2025,2025-11-20 12:14:00,ACRL,22682,5371,23.70,624,11.62
4,Syllabus - October 2025,The Syllabus From ACRL - October 2025,2025-10-15 12:13:00,ACRL,22775,4938,21.70,596,12.10
5,Syllabus - September 2025,The Syllabus From ACRL - September 2025,2025-09-17 14:38:00,ACRL,22638,5196,23.00,489,9.40
6,Syllabus - August 2025,The Syllabus From ACRL - August 2025,2025-08-20 12:12:00,ACRL,22725,8143,35.83,535,6.60
7,Syllabus - July 2025,The Syllabus From ACRL - July 2025,2025-07-16 12:14:00,ACRL,22793,8791,38.60,623,7.10
8,Syllabus - June 2025,The Syllabus From ACRL - June 2025,2025-06-18 14:40:00,ACRL,22797,6302,27.60,926,7.16
9,Syllabus - May 2025,The Syllabus From ACRL - May 2025,2025-05-21 12:13:00,ACRL,22797,6302,27.60,700,11.11


<hr>

## 3.0 ACRL eBlast Performance (Informz)

### 3.1 Individual Book Promos vs. Syllabus Newsletter — Summary

In [8]:
def email_summary(df, label):
    return pd.Series({
        'Source': label,
        'N E-Blasts':        len(df),
        'Avg # Delivered':   round(df['# Delivered'].mean()),
        'Avg Open Rate (%)': round(df['Open %'].mean(), 2),
        'Avg Click Rate (%)':round(df['Click %'].mean(), 2),
    })

acrl_email_summary = pd.DataFrame([
    email_summary(acrl_indiv_emails, 'ACRL Individual Book Promos'),
    email_summary(acrl_syllabus,     'ACRL Syllabus Newsletter'),
]).set_index('Source')

display(acrl_email_summary)

,N E-Blasts,Avg # Delivered,Avg Open Rate (%),Avg Click Rate (%)
Source,,,,
ACRL Individual Book Promos,19,18023,29.78,9.16
ACRL Syllabus Newsletter,21,22526,23.65,11.83


### 3.2 Open Rate & Click Rate — Individual Promos vs. Syllabus

In [9]:
data_types = {
    "Individual Book Promos": acrl_indiv_emails,
    "Syllabus Newsletter": acrl_syllabus
}

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Map labels to our brand colors
# Using Teal for Indiv Promos and Blue for Syllabus
color_map = {
    "Individual Book Promos": ACRL_TEAL, 
    "Syllabus Newsletter": ALA_BLUE
}

metrics = [
    {"key": "Open %", "label": "Open Rate (%)"},
    {"key": "Click %", "label": "Click Rate (%)"}
]

# 2. Create Subplots
fig = make_subplots(rows=1, cols=2, subplot_titles=[f"<b>{m['label']} Distribution</b>" for m in metrics])

for i, metric in enumerate(metrics, 1):
    for label, df in data_types.items():
        data = df[metric["key"]].dropna()
        
        fig.add_trace(
            go.Box(
                y=data,
                name=label,
                marker=dict(color=color_map[label]),
                fillcolor=color_map[label],
                opacity=0.7,
                boxpoints='outliers',
                # This 'line' property controls the median AND the box border
                line=dict(color='black', width=1.5), 
                marker_size=4
            ),
            row=1, col=i
        )

# 3. Update Layout & Styling
fig.update_layout(
    title_text="<b>ACRL Email Performance: Distribution by Type</b>",
    title_x=0.5,
    showlegend=False,
    template="plotly_white", # Keeps the background clean
    height=500,
    width=1000,
    margin=dict(t=80, b=50, l=50, r=50)
)

# Apply gray gridlines for a softer look
fig.update_xaxes(showline=True, linewidth=1, linecolor='lightgray', mirror=True)
fig.update_yaxes(showline=True, linewidth=1, linecolor='lightgray', mirror=True, gridcolor='whitesmoke')

fig.show()

### 3.3 ACRL Individual Book Promo — Top Performers

In [10]:
top_acrl = acrl_indiv_emails.nlargest(10, 'Open %')[['Subject', 'Sent', '# Delivered', 'Open %', 'Click %']]
display(top_acrl.reset_index(drop=True))

,Subject,Sent,# Delivered,Open %,Click %
0,"New from ACRL - ""The Open Science Cookbook""",2025-06-11 11:14:00,18217,43.0,8.8
1,"New from ACRL - ""Supporting Neurodiverse Stude...",2025-06-24 11:14:00,18420,41.7,4.6
2,"New from ACRL - ""Legislative Advocacy and Publ...",2025-05-22 11:14:00,18295,39.0,13.4
3,"New from ACRL - ""Valuing the Community College...",2025-07-30 11:14:00,18314,38.1,3.2
4,"New from ACRL - ""Library Publishing: How to La...",2025-10-29 11:14:00,18350,33.4,4.0
5,"New from ACRL - ""Making Values-Based Decisions...",2025-09-16 11:14:00,18211,32.7,4.5
6,"New from ACRL - ""Navigating Disability in the ...",2025-10-09 11:14:00,18408,32.6,4.9
7,"New from ACRL - ""Training Library Instructors""...",2024-10-08 11:12:00,17664,29.8,6.1
8,"New from ACRL - ""From Interrogation to Integra...",2025-04-09 11:14:00,18153,29.1,4.5
9,"New from ACRL - ""Student Success Librarianship...",2025-03-11 11:14:00,17982,29.0,10.7


### 3.4 Syllabus Newsletter: Trend Over Time

In [11]:
import plotly.graph_objects as go

# 1. Sort the data by date (crucial for trend lines)
syll_sorted = acrl_syllabus.sort_values('Sent')

# 2. Create figure with secondary y-axis
from plotly.subplots import make_subplots
fig = make_subplots(specs=[[{"secondary_y": True}]])

# 3. Add Open Rate Trace (Primary Axis)
fig.add_trace(
    go.Scatter(
        x=syll_sorted['Sent'],
        y=syll_sorted['Open %'],
        name="Open Rate (%)",
        mode='lines+markers',
        line=dict(color=ACRL_TEAL, width=3),
        marker=dict(size=8)
    ),
    secondary_y=False,
)

# 4. Add Click Rate Trace (Secondary Axis)
fig.add_trace(
    go.Scatter(
        x=syll_sorted['Sent'],
        y=syll_sorted['Click %'],
        name="Click Rate (%)",
        mode='lines+markers',
        line=dict(color=ALA_BLUE, width=2, dash='dash'), # Dashed line for distinction
        marker=dict(size=6, symbol='square')
    ),
    secondary_y=True,
)

# 5. Update Layout & Axes
fig.update_layout(
    title_text="<b>ACRL Syllabus Newsletter: Open & Click Rate Trends</b>",
    title_x=0.5,
    template="plotly_white",
    hovermode="x unified", # Shows both metrics in one tooltip when hovering
    width=1000,
    height=400,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Set x-axis title
fig.update_xaxes(title_text="Send Date")

# Set y-axes titles
fig.update_yaxes(title_text="<b>Open Rate (%)</b>", color=ACRL_TEAL, secondary_y=False)
fig.update_yaxes(title_text="<b>Click Rate (%)</b>", color=ALA_BLUE, secondary_y=True)

fig.show()

### 3.5 eBlast Open Rate vs. Click Rate (Individual Book Promotions)

In [12]:
import plotly.graph_objects as go

# 1. Create the figure
fig = go.Figure()

# 2. Add Individual Book Promos (Teal Bubbles)
fig.add_trace(
    go.Scatter(
        x=acrl_indiv_emails['Open %'],
        y=acrl_indiv_emails['Click %'],
        name='Individual Book Promo',
        mode='markers',
        text=acrl_indiv_emails['Subject'], 
        marker=dict(
            color=ACRL_TEAL,
            # Reduced from (* 40 + 10) to (* 25 + 8)
            size=acrl_indiv_emails['# Delivered'] / acrl_indiv_emails['# Delivered'].max() * 20 + 8,
            opacity=0.7,
            line=dict(color='white', width=1)
        ),
        hovertemplate="<b>%{text}</b><br>Open Rate: %{x}%<br>Click Rate: %{y}%<extra></extra>"
    )
)

# 3. Add Syllabus Newsletter (Blue Diamonds)
fig.add_trace(
    go.Scatter(
        x=acrl_syllabus['Open %'],
        y=acrl_syllabus['Click %'],
        name='Syllabus Newsletter',
        mode='markers',
        text=acrl_syllabus['Subject'],
        marker=dict(
            color=ALA_BLUE,
            size=12,
            symbol='diamond',
            opacity=0.8,
            line=dict(color='white', width=1)
        ),
        hovertemplate="<b>%{text}</b><br>Open Rate: %{x}%<br>Click Rate: %{y}%<extra></extra>"
    )
)

# 4. Add Median Reference Lines (Gray dashed)
indiv_open_med = acrl_indiv_emails['Open %'].median()
indiv_click_med = acrl_indiv_emails['Click %'].median()

fig.add_vline(x=indiv_open_med, line=dict(color="gray", width=1, dash="dot"))
fig.add_hline(y=indiv_click_med, line=dict(color="gray", width=1, dash="dot"))

# 5. Layout Styling
fig.update_layout(
    title="<b>ACRL E-Blasts: Open Rate vs. Click Rate</b><br><sup>Bubble size indicates volume delivered</sup>",
    xaxis_title="Open Rate (%)",
    yaxis_title="Click Rate (%)",
    template="plotly_white",
    width=900,
    height=600,
    # Updated Legend Position
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99,
        bgcolor="rgba(255, 255, 255, 0.7)", # Slight background for readability
        bordercolor="lightgray",
        borderwidth=1
    )
)

fig.show()

### 3.6 ACRL Emails: Open % and Click % by Send Date

In [13]:
import plotly.graph_objects as go

# 1. Create the figure
fig = go.Figure()

# 2. Add Open % (Teal with Fill)
fig.add_trace(
    go.Scatter(
        x=acrl_all_emails['Sent'],
        y=acrl_all_emails['Open %'],
        name='Open Rate (%)',
        mode='lines+markers',
        line=dict(color=ACRL_TEAL, width=2),
        marker=dict(size=6, symbol='circle'),
        fill='tozeroy',  # Replicates ax.fill_between
        fillcolor='rgba(0, 133, 124, 0.15)', # Teal with 15% opacity
        hovertemplate="<b>Date:</b> %{x|%b %d, %Y}<br><b>Open Rate:</b> %{y}%<extra></extra>"
    )
)

# 3. Add Click % (Blue Dashed)
fig.add_trace(
    go.Scatter(
        x=acrl_all_emails['Sent'],
        y=acrl_all_emails['Click %'],
        name='Click Rate (%)',
        mode='lines+markers',
        line=dict(color=ALA_BLUE, width=1.5, dash='dash'),
        marker=dict(size=5, symbol='square'),
        hovertemplate="<b>Date:</b> %{x|%b %d, %Y}<br><b>Click Rate:</b> %{y}%<extra></extra>"
    )
)

# 4. Update Layout & Axes
fig.update_layout(
    title_text="<b>ACRL Emails — Open % & Click % by Send Date</b>",
    title_x=0.5,
    xaxis_title="Date of Email Sent",
    yaxis_title="Engagement Rate (%)",
    template="plotly_white",
    xaxis=dict(
        tickformat="%b %d, %Y", # Result: "Jan 01, 2025"
        dtick="M1",              # Forces a tick every 1 month
        tickangle=-45            # Tilts for better readability
    ),
    hovermode="x unified", # Shows both metrics in one tooltip
    height=500,
    width=1000,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

# Improve x-axis date formatting
fig.update_xaxes(
    tickangle=-90,
    tickformat="%b %d %Y",
    dtick="M1", # Show ticks monthly
    showgrid=True,
    gridcolor='whitesmoke'
)

fig.show()

<hr>

## 4.0 ACRL Email Body Text Analysis

In [14]:
df = acrl_indiv_emails[acrl_indiv_emails["group"].notna()].copy()
high = df[df["group"] == "high"]
low  = df[df["group"] == "low"]

# ── 1. Word frequency diff table (printed, no plot needed) ────────────────────
def word_freq(texts):
    vec = CountVectorizer(stop_words="english", ngram_range=(1, 2))
    matrix = vec.fit_transform(texts)
    counts = matrix.sum(axis=0).A1
    return pd.Series(counts, index=vec.get_feature_names_out())

high_freq = word_freq(high["body_text"])
low_freq  = word_freq(low["body_text"])

freq_table = pd.DataFrame({"high": high_freq, "low": low_freq}).fillna(0).astype(int)
freq_table["diff"] = freq_table["high"] - freq_table["low"]
freq_table = freq_table.sort_values("diff", ascending=False)

print("── Words most associated with HIGH performers ──")
print(freq_table.head(20).to_string())
print("\n── Words most associated with LOW performers ──")
print(freq_table.tail(20).sort_values("diff").to_string())

# ── 2. TF-IDF bar charts ──────────────────────────────────────────────────────
def tfidf_top_words(texts, n=15):
    vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=500)
    matrix = vec.fit_transform(texts)
    scores = matrix.mean(axis=0).A1
    top_idx = scores.argsort()[::-1][:n]
    vocab = vec.get_feature_names_out()
    return pd.DataFrame({"term": vocab[top_idx], "score": scores[top_idx].round(4)})

high_tfidf = tfidf_top_words(high["body_text"])
low_tfidf  = tfidf_top_words(low["body_text"])

fig_tfidf = make_subplots(rows=1, cols=2,
                           subplot_titles=("TF-IDF: High Performers", "TF-IDF: Low Performers"))
fig_tfidf.add_trace(
    go.Bar(x=high_tfidf["score"], y=high_tfidf["term"],
           orientation="h", marker_color="steelblue", name="High"),
    row=1, col=1
)
fig_tfidf.add_trace(
    go.Bar(x=low_tfidf["score"], y=low_tfidf["term"],
           orientation="h", marker_color="tomato", name="Low"),
    row=1, col=2
)
fig_tfidf.update_yaxes(autorange="reversed")
fig_tfidf.update_layout(title="TF-IDF: High vs. Low Performers",
                         height=500, width=1700, showlegend=False)
fig_tfidf.show()

# ── 3. Subject line features ──────────────────────────────────────────────────
df["subj_char_len"]     = df["Subject_clean"].str.len()
df["subj_word_count"]   = df["Subject_clean"].str.split().str.len()
df["subj_has_subtitle"] = df["Subject_clean"].str.contains(r":", na=False).astype(int)

print("\n── Subject line features by group ──")
print(df.groupby("group")[["subj_char_len", "subj_word_count",
                            "subj_has_subtitle", "Open %", "Click %"]].mean().round(2))

# ── 4. Body length vs engagement ─────────────────────────────────────────────
df["body_word_count"] = df["body_text"].str.split().str.len()
df["short_name"]      = df["Name"].str.replace(" Book", "").str.replace(" Cookbook", "")

fig_body = make_subplots(rows=1, cols=2,
                          subplot_titles=("Body Length vs. Open %", "Body Length vs. Click %"))
for col, metric in enumerate(["Open %", "Click %"], start=1):
    for group, color in [("high", "steelblue"), ("low", "tomato")]:
        subset = df[df["group"] == group]
        fig_body.add_trace(
            go.Scatter(
                x=subset["body_word_count"],
                y=subset[metric],
                mode="markers+text",
                text=subset["short_name"],
                textposition="top center",
                textfont=dict(size=9),
                marker=dict(color=color, size=10, line=dict(color="white", width=1)),
                name=group.title(),
                showlegend=(col == 1)
            ),
            row=1, col=col
        )
    # Trend line
    z = np.polyfit(df["body_word_count"], df[metric], 1)
    x_line = np.linspace(df["body_word_count"].min(), df["body_word_count"].max(), 100)
    fig_body.add_trace(
        go.Scatter(x=x_line, y=np.poly1d(z)(x_line),
                   mode="lines", line=dict(dash="dash", color="gray", width=1.2),
                   showlegend=False),
        row=1, col=col
    )

fig_body.update_layout(title="<b>ACRL Body Length vs. Engagement</b>", title_x=0.5,
                        height=500, legend_title="Group")
fig_body.update_xaxes(title_text="Body Word Count")
fig_body.show()

# ── 5. Summary stats ──────────────────────────────────────────────────────────
print("\n── Avg Open % and Click % by group ──")
print(df.groupby("group")[["Open %", "Click %"]].mean().round(2))


── Words most associated with HIGH performers ──
                  high  low  diff
academic            23    1    22
library             25    8    17
student             15    0    15
research            11    0    11
acrl                12    2    10
student success      9    0     9
open                10    1     9
advocacy             9    0     9
success             10    2     8
diversity            8    0     8
science              8    0     8
workers              9    1     8
collection           9    1     8
library workers      9    1     8
open science         8    0     8
legislative          8    0     8
work                 8    1     7
academic library     7    0     7
print                9    2     7
policy               7    0     7

── Words most associated with LOW performers ──
                         high  low  diff
community college           0   10   -10
community                   2   11    -9
college                     2   10    -8
college library         


── Subject line features by group ──
       subj_char_len  subj_word_count  subj_has_subtitle  Open %  Click %
group                                                                    
high            92.4             13.4                0.8   32.72     10.0
low            102.0             14.0                1.0   38.10      3.2



── Avg Open % and Click % by group ──
       Open %  Click %
group                 
high    32.72     10.0
low     38.10      3.2
